#Info

## SimpleRotaryEmbedding Documentation

The `SimpleRotaryEmbedding` class implements Rotary Positional Embeddings (RoPE) in a memory-efficient manner, particularly beneficial for environments with limited memory such as a T4 GPU.

**Purpose:**

Rotary Positional Embeddings are a type of positional encoding that enhances attention mechanisms in Transformer models. Instead of adding positional information to token embeddings directly, RoPE applies a rotation to the query (Q) and key (K) vectors based on their absolute position. This allows for relative position information to be implicitly encoded.

This specific implementation is *"without caching"*, meaning the inverse frequencies (`inv_freq`) are computed and applied on-the-fly for each forward pass. While this might involve a slight computational overhead compared to pre-computed and cached versions, it significantly reduces memory consumption, making it suitable for training larger models or on devices with less available memory.

**Initialization (`__init__`)**

```python
def __init__(self, config):
    super().__init__()
    self.dim = config.head_dim
    self.theta = config.rope_theta

    inv_freq = 1.0 / (self.theta ** (torch.arange(0, self.dim, 2).float() / self.dim))
    self.register_buffer("inv_freq", inv_freq, persistent=False)
```

*   `config`: An object (e.g., `SimpleLLMConfig`) containing model configuration parameters.
*   `self.dim`: The dimensionality of the attention head (`head_dim`) for which the RoPE is applied. This determines the size of the vectors being rotated.
*   `self.theta`: A hyperparameter that controls the base frequency for the rotary embeddings. A common value is `10000.0`.
*   `inv_freq`: This is a buffer registered with the module, storing the inverse frequencies. It's computed once during initialization. It's marked as `persistent=False` because it's a static tensor derived from `theta` and `dim` and doesn't need to be saved as part of the model's state dictionary if not directly used for parameter updates.
    The formula `1.0 / (self.theta ** (torch.arange(0, self.dim, 2).float() / self.dim))` generates a sequence of inverse frequencies. It creates `dim/2` unique frequencies, which are then used to generate cosine and sine components for rotation.

**Forward Pass (`forward`)**

```python
def forward(self, x, position_ids=None):
    batch_size, seq_len, num_heads, head_dim = x.shape
    device = x.device

    if position_ids is None:
        position_ids = torch.arange(seq_len, device=device).expand(batch_size, seq_len)

    # Compute frequencies on-the-fly
    t = position_ids.float().unsqueeze(-1)  # [batch_size, seq_len, 1]
    freqs = t * self.inv_freq  # [batch_size, seq_len, dim//2]

    # Expand to full dimension
    freqs = torch.cat([freqs, freqs], dim=-1)  # [batch_size, seq_len, dim]
    freqs = freqs.unsqueeze(2)  # [batch_size, seq_len, 1, dim]

    # Apply rotation
    cos = torch.cos(freqs)
    sin = torch.sin(freqs)

    x1 = x[..., : self.dim // 2]
    x2 = x[..., self.dim // 2 :]

    rotated_x1 = x1 * cos - x2 * sin
    rotated_x2 = x1 * sin + x2 * cos

    return torch.cat((rotated_x1, rotated_x2), dim=-1)
```

*   `x`: The input tensor, typically a query or key vector from an attention head, with shape `[batch_size, seq_len, num_heads, head_dim]`.
*   `position_ids`: Optional tensor of explicit position IDs for each token. If not provided, it defaults to `[0, 1, ..., seq_len-1]` for each sequence in the batch.

**Process:**

1.  **Shape Extraction:** Retrieves `batch_size`, `seq_len`, `num_heads`, and `head_dim` from the input tensor `x`.
2.  **Position ID Handling:** If `position_ids` is `None`, it generates a default sequence of position IDs from `0` to `seq_len - 1` for each item in the batch.
3.  **Frequency Computation:**
    *   `t`: The `position_ids` are converted to float and an extra dimension is added (`unsqueeze(-1)`), resulting in a shape like `[batch_size, seq_len, 1]`. This `t` represents the absolute position of each token.
    *   `freqs`: This is calculated by multiplying `t` with the pre-computed `self.inv_freq`. The `inv_freq` has shape `[dim//2]`, and broadcasting ensures `freqs` gets shape `[batch_size, seq_len, dim//2]`. These are the unique frequencies for the first half of the dimension.
4.  **Frequency Expansion:** The `freqs` tensor is concatenated with itself along the last dimension to match the full `head_dim`. Then, it's unsqueezed to add a dimension for `num_heads`, resulting in `[batch_size, seq_len, 1, dim]`. This prepares `freqs` for element-wise multiplication with `x`.
5.  **Rotation Application:**
    *   `cos` and `sin` components are derived from `freqs`.
    *   The input `x` is split into two halves: `x1` (first `dim/2`) and `x2` (second `dim/2`).
    *   The rotation is applied using the standard 2D rotation matrix formula: `x' = x cos - y sin` and `y' = x sin + y cos`.
    *   `rotated_x1` and `rotated_x2` are computed.
6.  **Concatenation:** The rotated halves (`rotated_x1`, `rotated_x2`) are concatenated back together to form the final rotated tensor, which is then returned.

# Model

##Dependencies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple, Dict, Any, Union, List, Iterator
import math
from pathlib import Path
import json
from datetime import datetime
import logging
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from torch.cuda.amp import GradScaler, autocast
from transformers import get_linear_schedule_with_warmup

In [ ]:
from dataclasses import dataclass, field
import math

@dataclass
class SimpleLLMConfig:
    # === Core Model Dimensions ===
    vocab_size: int = 50265
    """Total number of tokens in the vocabulary."""

    hidden_size: int = 512
    """Dimensionality of token embeddings and hidden states."""

    # === Transformer Architecture ===
    num_hidden_layers: int = 8
    """Number of Transformer decoder blocks."""

    num_attention_heads: int = 8
    """Total number of attention heads per layer."""

    num_key_value_heads: int = 4
    """Number of heads used for key/value projections."""

    head_dim: int = field(init=False)
    """Dimensionality of each attention head."""

    # === Feed-Forward Network ===
    intermediate_size: int = 2048
    """Size of the hidden layer in the feedforward network."""

    # === Positional Embeddings ===
    max_position_embeddings: int = 2048
    """Maximum number of positional embeddings."""

    rope_theta: float = 10000.0
    """Base frequency for Rotary Positional Embeddings."""

    # === Normalization & Regularization ===
    rms_norm_eps: float = 1e-6
    """Epsilon value for RMSNorm."""

    attention_dropout: float = 0.1
    """Dropout applied to attention weights."""

    residual_dropout: float = 0.1
    """Dropout applied to residual connections."""

    # === Advanced Features ===
    use_qk_norm: bool = True
    """Whether to normalize Q and K vectors in attention."""

    # === Memory Optimizations ===
    enable_gradient_checkpointing: bool = True
    """Enable memory-efficient training."""

    # === Tokenizer Settings ===
    pad_token_id: int = 1 # Reverted to 1 for input_ids padding
    """Token ID used for padding."""

    def __post_init__(self):
        if self.num_attention_heads % self.num_key_value_heads != 0:
            raise ValueError(
                f"num_attention_heads ({self.num_attention_heads}) must be divisible by num_key_value_heads "
                f"({self.num_key_value_heads})"
            )
        self.head_dim = self.hidden_size // self.num_attention_heads

        if self.head_dim * self.num_attention_heads != self.hidden_size:
            raise ValueError(
                f"hidden_size ({self.hidden_size}) is not divisible by num_attention_heads ({self.num_attention_heads})"
            )


In [ ]:
config = SimpleLLMConfig()
print(config)

SimpleLLMConfig(vocab_size=50265, hidden_size=512, num_hidden_layers=8, num_attention_heads=8, num_key_value_heads=4, head_dim=64, intermediate_size=2048, max_position_embeddings=2048, rope_theta=10000.0, rms_norm_eps=1e-06, attention_dropout=0.1, residual_dropout=0.1, use_qk_norm=True, enable_gradient_checkpointing=True, pad_token_id=1)


In [ ]:
import json
from dataclasses import asdict

# Ensure config object is created
config = SimpleLLMConfig()

# Convert dataclass to dictionary and pretty print as JSON
config_dict = asdict(config)
print(json.dumps(config_dict, indent=2))

{
  "vocab_size": 50265,
  "hidden_size": 512,
  "num_hidden_layers": 8,
  "num_attention_heads": 8,
  "num_key_value_heads": 4,
  "head_dim": 64,
  "intermediate_size": 2048,
  "max_position_embeddings": 2048,
  "rope_theta": 10000.0,
  "rms_norm_eps": 1e-06,
  "attention_dropout": 0.1,
  "residual_dropout": 0.1,
  "use_qk_norm": true,
  "enable_gradient_checkpointing": true,
  "pad_token_id": 1
}


## SimpleRotaryEmbedding

In [ ]:
class SimpleRotaryEmbedding(nn.Module):
    """
    Simple Rotary Embedding without caching - uses less memory.
    Better for training on T4 where memory is limited.
    """

    def __init__(self, config):
        super().__init__()
        self.dim = config.head_dim
        self.theta = config.rope_theta

        inv_freq = 1.0 / (self.theta ** (torch.arange(0, self.dim, 2).float() / self.dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x, position_ids=None):
        batch_size, seq_len, num_heads, head_dim = x.shape
        device = x.device

        if position_ids is None:
            position_ids = torch.arange(seq_len, device=device).expand(batch_size, seq_len)

        # Compute frequencies on-the-fly for dim//2
        t = position_ids.float().unsqueeze(-1)  # [batch_size, seq_len, 1]

        freqs = t * self.inv_freq  # [batch_size, seq_len, dim//2]

        # Reshape freqs to match x1/x2 for broadcasting: [batch_size, seq_len, 1, dim//2]
        # The '1' dimension is for num_heads, which will broadcast across all heads
        freqs = freqs.unsqueeze(2) # [batch_size, seq_len, 1, dim//2]

        # Apply rotation
        cos = torch.cos(freqs)  # [batch_size, seq_len, 1, dim//2]
        sin = torch.sin(freqs)  # [batch_size, seq_len, 1, dim//2]

        x1 = x[..., : self.dim // 2] # [batch_size, seq_len, num_heads, dim//2]
        x2 = x[..., self.dim // 2 :] # [batch_size, seq_len, num_heads, dim//2]

        rotated_x1 = x1 * cos - x2 * sin
        rotated_x2 = x1 * sin + x2 * cos

        return torch.cat((rotated_x1, rotated_x2), dim=-1)


##Rmsn

In [ ]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    """
    Root Mean Square Normalization (RMSNorm) Module.

    Normalizes input tensor across the last dimension using RMS,
    without subtracting the mean. Used in models like LLaMA for
    efficient and stable training.

    Args:
        hidden_size (int): Size of the last dimension to normalize.
        eps (float): Small constant to avoid division by zero.
    """
    def __init__(self, hidden_size, eps=1e-8):
        super(RMSNorm, self).__init__()
        self.hidden_size = hidden_size
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(hidden_size))  # Learnable scale

    def forward(self, x):
        """
        Apply RMS normalization to input tensor.

        Args:
            x (Tensor): Input of shape (..., hidden_size)

        Returns:
            Tensor: Normalized output of same shape as input.
        """
        # Compute RMS across last dimension
        rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        # Normalize and scale
        return x * rms * self.gamma


## GroupedQueryAttention

In [ ]:
class GroupedQueryAttention(nn.Module):
    """
    Grouped Query Attention (GQA) with your SimpleRotaryEmbedding.
    Optimized for T4 GPU training.
    """

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = config.head_dim
        self.max_seq_len = config.max_position_embeddings

        # Validate dimensions
        assert self.hidden_size % self.num_heads == 0, "hidden_size must be divisible by num_heads"
        assert self.num_heads % self.num_kv_heads == 0, "num_heads must be divisible by num_kv_heads"

        self.n_rep = self.num_heads // self.num_kv_heads  # repetition factor

        # Projection layers
        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)

        # Dropout
        self.attn_dropout = nn.Dropout(config.attention_dropout)
        self.resid_dropout = nn.Dropout(config.residual_dropout)

        # Your SimpleRotaryEmbedding
        self.rotary_emb = SimpleRotaryEmbedding(config)

        # Optional: QK normalization
        self.use_qk_norm = getattr(config, 'use_qk_norm', False)
        if self.use_qk_norm:
            self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
            self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)

    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None):
        """
        Forward pass for GQA with SimpleRotaryEmbedding.

        Args:
            x: Input tensor [batch_size, seq_len, hidden_size]
            attention_mask: Optional attention mask [batch_size, seq_len] (padding mask)

        Returns:
            attention_output: [batch_size, seq_len, hidden_size]
        """
        batch_size, seq_len, _ = x.shape
        device = x.device

        # Project queries, keys, values
        q = self.q_proj(x)  # [batch_size, seq_len, num_heads * head_dim]
        k = self.k_proj(x)  # [batch_size, seq_len, num_kv_heads * head_dim]
        v = self.v_proj(x)  # [batch_size, seq_len, num_kv_heads * head_dim]

        # Reshape to separate heads
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim)
        k = k.view(batch_size, seq_len, self.num_kv_heads, self.head_dim)
        v = v.view(batch_size, seq_len, self.num_kv_heads, self.head_dim)

        # Apply QK normalization if enabled
        if self.use_qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)

        # Apply rotary positional embeddings to both queries and keys
        q = self.rotary_emb(q)  # [batch_size, seq_len, num_heads, head_dim]
        k = self.rotary_emb(k)  # [batch_size, seq_len, num_kv_heads, head_dim]

        # Repeat k and v for grouped query attention
        k = k.repeat_interleave(self.n_rep, dim=2)  # [batch_size, seq_len, num_heads, head_dim]
        v = v.repeat_interleave(self.n_rep, dim=2)  # [batch_size, seq_len, num_heads, head_dim]

        # Transpose for attention computation
        q = q.transpose(1, 2)  # [batch_size, num_heads, seq_len, head_dim]
        k = k.transpose(1, 2)  # [batch_size, num_heads, seq_len, head_dim]
        v = v.transpose(1, 2)  # [batch_size, num_heads, seq_len, head_dim]

        # Compute attention scores
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim) # [batch_size, num_heads, seq_len, seq_len]

        # Apply attention mask (causal mask for training)
        # The incoming attention_mask is a padding mask [batch_size, seq_len]
        # We need to combine it with a causal mask.
        causal_mask = self._create_causal_mask(seq_len, device) # [1, 1, seq_len, seq_len]

        if attention_mask is not None:
            # Expand padding mask to [batch_size, 1, 1, seq_len] for broadcasting
            # Convert 0s (padding) to -inf and 1s (valid) to 0
            padding_mask_expanded = (1.0 - attention_mask.unsqueeze(1).unsqueeze(1)) * torch.finfo(scores.dtype).min

            # Combine causal and padding masks
            # The result will be [batch_size, 1, seq_len, seq_len]
            combined_mask = causal_mask + padding_mask_expanded
            scores = scores + combined_mask
        else:
            # Only apply causal mask if no padding mask is provided
            scores = scores + causal_mask

        # Softmax and dropout
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        # Apply attention to values
        attn_output = torch.matmul(attn_weights, v)  # [batch_size, num_heads, seq_len, head_dim]

        # Transpose back and combine heads
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, self.num_heads * self.head_dim)

        # Final projection and dropout
        output = self.o_proj(attn_output)
        output = self.resid_dropout(output)

        return output

    def _create_causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        """Create causal mask for autoregressive training."""
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]

SwiGLUFFN

In [ ]:
class SwiGLUFFN(nn.Module):
    """
    SwiGLU Feed-Forward Network (FFN) with gated linear units.
    More expressive and efficient than standard FFN with ReLU.

    Formula: SwiGLU(x) = (SiLU(xW_g) ⊗ xW_u) W_d
    Where ⊗ is element-wise multiplication.

    Args:
        config: Model configuration with hidden_size and intermediate_size
    """

    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size

        # Gate projection (W_g) - goes through SiLU activation
        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)

        # Up projection (W_u) - no activation
        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)

        # Down projection (W_d) - projects back to hidden size
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=False)

        # Dropout for regularization
        self.dropout = nn.Dropout(config.residual_dropout)

        # Initialize weights properly
        self._init_weights()

    def _init_weights(self):
        """Initialize weights like in LLaMA for better training stability."""
        # Standard initialization for linear layers
        nn.init.normal_(self.gate_proj.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.up_proj.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.down_proj.weight, mean=0.0, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for SwiGLU FFN.

        Args:
            x: Input tensor of shape [batch_size, seq_len, hidden_size]

        Returns:
            Output tensor of shape [batch_size, seq_len, hidden_size]
        """
        # SwiGLU: SiLU(gate) * up
        gate = F.silu(self.gate_proj(x))  # SiLU activation (also called Swish)
        up = self.up_proj(x)

        # Element-wise multiplication and dropout
        hidden = self.dropout(gate * up)

        # Project back to hidden size
        output = self.down_proj(hidden)

        return output

## TransformerBlock

In [ ]:
class TransformerBlock(nn.Module):
    """
    Single Transformer Block with:
    - Pre-norm architecture
    - Grouped Query Attention with Rotary Embeddings
    - SwiGLU Feed-Forward Network
    - Residual connections

    Args:
        config: Model configuration
    """

    def __init__(self, config):
        super().__init__()
        self.config = config

        # Self-attention path
        self.attn_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.attention = GroupedQueryAttention(config)

        # Feed-forward path
        self.ffn_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.ffn = SwiGLUFFN(config)

        # Residual dropout
        self.residual_dropout = nn.Dropout(config.residual_dropout)

    def forward(self,
                x: torch.Tensor,
                attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Forward pass for transformer block.

        Args:
            x: Input tensor [batch_size, seq_len, hidden_size]
            attention_mask: Optional attention mask

        Returns:
            Output tensor [batch_size, seq_len, hidden_size]
        """
        # === Self-Attention Path ===
        residual = x
        x_norm = self.attn_norm(x)
        attn_output = self.attention(x_norm, attention_mask)
        x = residual + self.residual_dropout(attn_output)

        # === Feed-Forward Path ===
        residual = x
        x_norm = self.ffn_norm(x)
        ffn_output = self.ffn(x_norm)
        x = residual + self.residual_dropout(ffn_output)

        return x

##MultiTransformerBlock

In [ ]:
class MultiTransformerBlock(nn.Module):
    """
    Stack of multiple transformer blocks forming the core of the LLM.

    Args:
        config: Model configuration
    """

    def __init__(self, config):
        super().__init__()
        self.config = config

        # Create all transformer layers
        self.layers = nn.ModuleList([
            TransformerBlock(config) for _ in range(config.num_hidden_layers)
        ])

        # Final normalization after all layers
        self.final_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

        # Gradient checkpointing for memory efficiency
        self.gradient_checkpointing = getattr(config, 'enable_gradient_checkpointing', False)

    def forward(self,
                hidden_states: torch.Tensor,
                attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Forward pass through all transformer blocks.

        Args:
            hidden_states: Input tensor [batch_size, seq_len, hidden_size]
            attention_mask: Optional attention mask

        Returns:
            Final hidden states after all layers
        """
        # Apply all transformer layers
        for layer in self.layers:
            if self.gradient_checkpointing and self.training:
                # Use gradient checkpointing to save memory
                hidden_states = torch.utils.checkpoint.checkpoint(
                    layer, hidden_states, attention_mask, use_reentrant=False
                )
            else:
                hidden_states = layer(hidden_states, attention_mask)

        # Apply final normalization
        hidden_states = self.final_norm(hidden_states)

        return hidden_states

## EmbeddingLayer

In [ ]:
class EmbeddingLayer(nn.Module):
    """Your existing embedding layer"""
    def __init__(self, vocab_size, hidden_dim, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_dim,
            sparse=True,  # Sparse gradients save memory
            padding_idx=1
        )
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(hidden_dim)

    def forward(self, input_ids):
        return self.dropout(self.token_embedding(input_ids) * self.scale)

## MinimalOutputHead

In [ ]:
class MinimalOutputHead(nn.Module):
    """Your perfect output head"""
    def __init__(self, hidden_size: int, vocab_size: int, embed_weight: nn.Parameter):
        super().__init__()
        self.norm = nn.RMSNorm(hidden_size)
        self.head = nn.Linear(hidden_size, vocab_size, bias=False)
        self.head.weight = embed_weight  # Weight tying

    def forward(self, x):
        return self.head(self.norm(x))

## SImple llm

In [ ]:
class SimpleLLM(nn.Module):
    """Complete LLM using your components"""
    def __init__(self, config):
        super().__init__()
        self.config = config

        # Embedding layer
        self.embedding = EmbeddingLayer(
            vocab_size=config.vocab_size,
            hidden_dim=config.hidden_size,
            dropout=config.residual_dropout
        )

        # Your transformer blocks
        self.transformer = MultiTransformerBlock(config)

        # Your minimal output head with weight tying
        self.output_head = MinimalOutputHead(
            hidden_size=config.hidden_size,
            vocab_size=config.vocab_size,
            embed_weight=self.embedding.token_embedding.weight
        )

        print(f"✓ Model ready for T4: {self.get_param_count() / 1e6:.1f}M params")

    def get_param_count(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, input_ids, attention_mask=None, labels=None):
        x = self.embedding(input_ids)

        # Transformer layers
        x = self.transformer(x, attention_mask=attention_mask)

        # Output logits
        logits = self.output_head(x)

        # Loss if training
        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),
                labels.view(-1),
                ignore_index=-100 # Explicitly ignore -100 for labels padding
            )

        return {'logits': logits, 'loss': loss}

In [ ]:
def get_model_memory_usage(model):
    """Calculates the memory usage of a PyTorch model in MB."""
    mem_params = sum(p.numel() * p.element_size() for p in model.parameters())
    mem_buffers = sum(buf.numel() * buf.element_size() for buf in model.buffers())
    mem_total = mem_params + mem_buffers
    print(f"Model memory usage: {mem_total / (1024**2):.2f} MB")
    return mem_total

#model =  SimpleLLM(config)
#get_model_memory_usage(model)

## Data loadeing

In [ ]:
class SimpleBatchCache:
    """
    Simple batch cache with master.json metadata tracking.
    Stores batches as batch001.pt, batch002.pt, etc.
    Tracks metadata in master.json for easy monitoring.
    """

    def __init__(self, cache_dir: str = "./batch_cache"):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)

        # Master metadata file
        self.master_file = self.cache_dir / "master.json"
        self.metadata = self._load_metadata()

        # Find the next available batch number
        self.next_batch_num = self._get_next_batch_number()

    def _load_metadata(self) -> Dict[str, Any]:
        """Load or initialize master metadata."""
        if self.master_file.exists():
            with open(self.master_file, 'r') as f:
                return json.load(f)
        else:
            # Initialize fresh metadata
            return {
                "created_at": datetime.now().isoformat(),
                "last_updated": datetime.now().isoformat(),
                "total_files": 0,
                "total_batches": 0,
                "files": {},
                "stats": {
                    "total_tokens": 0,
                    "total_sequences": 0,
                    "avg_sequence_length": 0
                }
            }

    def _save_metadata(self):
        """Save metadata to master.json."""
        self.metadata["last_updated"] = datetime.now().isoformat()
        with open(self.master_file, 'w') as f:
            json.dump(self.metadata, f, indent=2)

    def _get_next_batch_number(self) -> int:
        """Find the highest batch number from metadata or files."""
        if self.metadata["files"]:
            return max(int(k) for k in self.metadata["files"].keys()) + 1
        else:
            # Fallback to file scanning if metadata is empty
            existing_files = list(self.cache_dir.glob("batch*.pt"))
            if not existing_files:
                return 1
            numbers = []
            for file in existing_files:
                try:
                    num_str = file.stem.replace("batch", "")
                    numbers.append(int(num_str))
                except ValueError:
                    continue
            return max(numbers) + 1 if numbers else 1

    def _calculate_batch_stats(self, batches: List[Dict[str, torch.Tensor]]) -> Dict[str, Any]:
        """Calculate statistics for a set of batches."""
        total_tokens = 0
        total_sequences = 0

        for batch in batches:
            if 'input_ids' in batch:
                batch_size, seq_len = batch['input_ids'].shape
                total_sequences += batch_size
                total_tokens += batch_size * seq_len

        return {
            "num_batches": len(batches),
            "total_sequences": total_sequences,
            "total_tokens": total_tokens,
            "avg_sequence_length": total_tokens / total_sequences if total_sequences > 0 else 0
        }

    def save_batches(self, batches: List[Dict[str, torch.Tensor]],
                    metadata: Dict[str, Any] = None) -> str:
        """
        Save batches with metadata tracking.

        Args:
            batches: List of batch dictionaries
            metadata: Additional metadata about these batches

        Returns:
            Path to saved file
        """
        filename = f"batch{self.next_batch_num:03d}.pt"
        filepath = self.cache_dir / filename

        # Save batches to file
        torch.save(batches, filepath)

        # Calculate batch statistics
        batch_stats = self._calculate_batch_stats(batches)

        # Update master metadata
        file_metadata = {
            "filename": filename,
            "created_at": datetime.now().isoformat(),
            "num_batches": len(batches),
            **batch_stats
        }

        # Add custom metadata if provided
        if metadata:
            file_metadata["custom"] = metadata

        # Update master metadata
        self.metadata["files"][str(self.next_batch_num)] = file_metadata
        self.metadata["total_files"] = len(self.metadata["files"])
        self.metadata["total_batches"] += len(batches)

        # Update global stats
        self.metadata["stats"]["total_tokens"] += batch_stats["total_tokens"]
        self.metadata["stats"]["total_sequences"] += batch_stats["total_sequences"]
        if self.metadata["stats"]["total_sequences"] > 0:
            self.metadata["stats"]["avg_sequence_length"] = (
                self.metadata["stats"]["total_tokens"] /
                self.metadata["stats"]["total_sequences"]
            )

        # Save metadata
        self._save_metadata()

        print(f"💾 Saved {len(batches)} batches to {filename}")
        print(f"📊 Stats: {batch_stats['total_sequences']} sequences, {batch_stats['total_tokens']} tokens")

        # Increment for next file
        self.next_batch_num += 1

        return str(filepath)

    def save_single_batch(self, batch: Dict[str, torch.Tensor],
                         metadata: Dict[str, Any] = None) -> str:
        """Save a single batch with metadata."""
        return self.save_batches([batch], metadata)

    def load_batches(self, filename: str) -> List[Dict[str, torch.Tensor]]:
        """Load batches from a file."""
        return torch.load(self.cache_dir / filename)

    def get_file_metadata(self, batch_num: int) -> Dict[str, Any]:
        """Get metadata for a specific batch file."""
        return self.metadata["files"].get(str(batch_num), {})

    def get_all_files(self) -> List[str]:
        """Get all batch files in sequential order with metadata."""
        files = []
        for batch_num in sorted(self.metadata["files"].keys(), key=int):
            file_info = self.metadata["files"][batch_num]
            files.append(file_info["filename"])
        return files

    def get_all_files_with_metadata(self) -> List[Dict[str, Any]]:
        """Get all files with their metadata."""
        return [
            {**self.metadata["files"][batch_num], "batch_number": int(batch_num)}
            for batch_num in sorted(self.metadata["files"].keys(), key=int)
        ]

    def iter_all_batches(self) -> Iterator[Dict[str, torch.Tensor]]:
        """Iterate through all batches in all files."""
        for file_path in self.get_all_files():
            batches = self.load_batches(file_path)
            for batch in batches:
                yield batch

    def iter_files_with_metadata(self) -> Iterator[tuple[List[Dict[str, torch.Tensor]], Dict[str, Any]]]:
        """Iterate through files with their metadata."""
        for file_info in self.get_all_files_with_metadata():
            batches = self.load_batches(file_info["filename"])
            yield batches, file_info

    def clear_cache(self):
        """Clear all cached files and reset metadata."""
        for file in self.cache_dir.glob("batch*.pt"):
            file.unlink()

        if self.master_file.exists():
            self.master_file.unlink()

        # Reset
        self.metadata = self._load_metadata()  # This will create fresh metadata
        self.next_batch_num = 1

        print("🧹 Batch cache cleared")

    def get_stats(self) -> Dict[str, Any]:
        """Get comprehensive cache statistics."""
        files = self.get_all_files()
        total_batches = self.metadata["total_batches"]

        return {
            **self.metadata["stats"],
            "total_files": len(files),
            "total_batches": total_batches,
            "cache_dir": str(self.cache_dir),
            "created_at": self.metadata["created_at"],
            "last_updated": self.metadata["last_updated"]
        }

    def print_detailed_stats(self):
        """Print detailed statistics about the cache."""
        stats = self.get_stats()
        files_info = self.get_all_files_with_metadata()

        print("=" * 50)
        print("BATCH CACHE STATISTICS")
        print("=" * 50)
        print(f"Total files: {stats['total_files']}")
        print(f"Total batches: {stats['total_batches']}")
        print(f"Total sequences: {stats['total_sequences']:,}")
        print(f"Total tokens: {stats['total_tokens']:,}")
        print(f"Avg sequence length: {stats['avg_sequence_length']:.1f}")
        print(f"Cache directory: {stats['cache_dir']}")
        print(f"Created: {stats['created_at']}")
        print(f"Last updated: {stats['last_updated']}")
        print("\nFile Details:")
        for file_info in files_info:
            print(f"  {file_info['filename']}: {file_info['num_batches']} batches, "
                  f"{file_info['total_sequences']} sequences, {file_info['total_tokens']} tokens")
        print("=" * 50)

    def update_file_metadata(self, batch_num: int, new_metadata: Dict[str, Any]):
        """Update metadata for a specific batch file."""
        if str(batch_num) in self.metadata["files"]:
            self.metadata["files"][str(batch_num)].update(new_metadata)
            self._save_metadata()

# TRAINING

## model

In [ ]:
config = SimpleLLMConfig()
model =  SimpleLLM(config)

✓ Model ready for T4: 57.2M params


## Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_config = {
    'model_folder': '/content/drive/MyDrive/checkpoints',
    'model_basename': 'llm_',
    'preload': 'epoch23',
    'num_epochs': 24,
    'lr': 3e-4,
    'betas': (0.9, 0.97),
    'eps': 1e-8,
    'weight_decay': 0.01,
    'warmup_steps': 1000,
    'max_grad_norm': 1.0,
    'log_frequency': 100,
    'use_amp': True,
    'experiment_name': 'llm_training',
    'log_dir': '/content/drive/MyDrive/training_logs',
    'batch_files':['batch075.pt','batch076.pt',],
    'gradient_accumulation_steps': 4
    }

Logging

In [ ]:
def setup_logging(log_dir: str = "./logs"):
    """Setup logging configuration."""
    log_dir = Path(log_dir)
    log_dir.mkdir(exist_ok=True)

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_dir / f"training_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

Get File Path

In [ ]:
def get_weights_file_path(train_config, epoch: int) -> Path:
    """Generate model checkpoint file path."""
    model_folder = Path(train_config['model_folder'])
    model_folder.mkdir(exist_ok=True)
    model_basename = train_config['model_basename']
    filename = f"{model_basename}epoch{epoch}.pt"
    return model_folder / filename


save_checkpoint

In [ ]:
def save_checkpoint(train_config, epoch, model, optimizer, scheduler, scaler, global_step, loss):
    """Save training checkpoint."""
    model_filename = get_weights_file_path(train_config, epoch)

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'scaler_state_dict': scaler.state_dict() if scaler else None,
        'global_step': global_step,
        'loss': loss,
        'train_config': train_config,
        'timestamp': datetime.now().isoformat()
    }

    torch.save(checkpoint, model_filename)
    return model_filename

In [ ]:
def load_checkpoint(train_config, device):
    """Load training checkpoint if preload is specified."""
    if not train_config.get('preload'):
        return None, 0, 0

    try:
        preload_epoch = int(train_config['preload'].split("epoch")[-1])
        model_filename = get_weights_file_path(train_config, preload_epoch)

        if not model_filename.exists():
            logging.warning(f"Preload file {model_filename} not found. Starting from scratch.")
            return None, 0, 0

        logging.info(f"Preloading model: {model_filename}")
        checkpoint = torch.load(model_filename, map_location=device)
        return checkpoint, checkpoint['epoch'], checkpoint['global_step']

    except Exception as e:
        logging.error(f"Error loading checkpoint: {e}")
        return None, 0, 0

constant learning

In [ ]:
class SimpleLLMTrainer:
    """Trainer for SimpleLLM model with cached batches."""

    def __init__(self, model, cache, train_config, device=None):
        self.model = model
        self.cache = cache
        self.train_config = train_config
        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # Setup components
        self.logger = setup_logging(train_config.get('log_dir', './logs'))
        self.writer = SummaryWriter(train_config.get('experiment_name', 'llm_experiment'))

        # Move model to device
        self.model.to(self.device)
        self.logger.info(f"Using device: {self.device}")

        # Training stats
        self.global_step = 0
        self.epoch = 0

    def get_training_files(self):
        """Get specific batch files to use for training based on config."""
        if 'batch_files' in self.train_config:
            # User specified exact file names
            specified_files = self.train_config['batch_files']
            available_files = self.cache.get_all_files()

            # Filter to only include files that exist
            training_files = []
            for file_name in specified_files:
                file_path = self.cache.cache_dir / file_name
                if file_path.exists():
                    training_files.append(str(file_path))
                else:
                    self.logger.warning(f"Specified batch file not found: {file_name}")

            if not training_files:
                self.logger.error("No specified batch files found! Using all available files.")
                training_files = available_files
            else:
                self.logger.info(f"Using {len(training_files)} specified batch files")

        else:
            # Use all available files (original behavior)
            training_files = self.cache.get_all_files()
            self.logger.info(f"Using all {len(training_files)} available batch files")

        return training_files

    def setup_training(self):
        """Setup optimizer, scheduler, and mixed precision."""
        # Optimizer
        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=self.train_config.get('lr', 1e-4),
            betas=self.train_config.get('betas', (0.9, 0.999)),
            eps=self.train_config.get('eps', 1e-8),
            weight_decay=self.train_config.get('weight_decay', 0.01)
        )

        # Get training files and calculate total steps
        self.training_files = self.get_training_files()
        total_batches_raw = sum(len(self.cache.load_batches(f)) for f in self.training_files)
        num_epochs = self.train_config.get('num_epochs', 3)
        grad_acc_steps = self.train_config.get('gradient_accumulation_steps', 1)

        # Total optimization steps (not total micro-batches)
        self.total_steps = (total_batches_raw * num_epochs) // grad_acc_steps
        if (total_batches_raw * num_epochs) % grad_acc_steps != 0:
            self.total_steps += 1 # Account for partial accumulation at the end

        # Reintroducing learning rate scheduler
        warmup_steps = self.train_config.get('warmup_steps', 1000)
        self.scheduler = get_linear_schedule_with_warmup(
            self.optimizer, num_warmup_steps=warmup_steps, num_training_steps=self.total_steps
        )

        # Mixed precision
        self.scaler = GradScaler(enabled=self.train_config.get('use_amp', True))

        # Gradient clipping
        self.max_grad_norm = self.train_config.get('max_grad_norm', 1.0)
        self.grad_acc_steps = grad_acc_steps # Store for easy access

        self.logger.info(f"Total optimization steps: {self.total_steps}")
        self.logger.info(f"Total micro-batches per epoch: {total_batches_raw}")
        self.logger.info(f"Gradient accumulation steps: {self.grad_acc_steps}")
        self.logger.info(f"Warmup steps: {warmup_steps}")
        self.logger.info(f"Training files: {[Path(f).name for f in self.training_files]}")

    def load_checkpoint(self):
        """Load checkpoint if specified in config."""
        checkpoint, epoch, global_step = load_checkpoint(self.train_config, self.device)
        if checkpoint:
            self.model.load_state_dict(checkpoint['model_state_dict'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            if self.scheduler and checkpoint['scheduler_state_dict']:
                self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            if self.scaler and checkpoint['scaler_state_dict']:
                self.scaler.load_state_dict(checkpoint['scaler_state_dict'])

            self.epoch = checkpoint['epoch']
            self.global_step = checkpoint['global_step']
            self.logger.info(f"Resumed from epoch {self.epoch}, global step {self.global_step}")

    def calculate_accuracy(self, logits, labels):
        """Calculate accuracy ignoring padding tokens."""
        # Flatten the tensors
        logits_flat = logits.view(-1, logits.size(-1))
        labels_flat = labels.view(-1)

        # Only consider non-padding tokens (labels != -100)
        non_pad_mask = labels_flat != -100
        if non_pad_mask.sum() == 0:
            return 0.0

        # Get predictions
        preds = torch.argmax(logits_flat, dim=-1)

        # Calculate accuracy on non-padding tokens
        correct = (preds[non_pad_mask] == labels_flat[non_pad_mask]).float()
        accuracy = correct.mean().item()

        return accuracy

    def train_epoch(self, epoch):
        """Train for one epoch."""
        self.model.train()
        total_loss = 0
        total_accuracy = 0
        total_perplexity = 0 # New: for perplexity
        micro_batch_count = 0
        accumulated_loss = 0
        accumulated_accuracy = 0
        accumulated_perplexity = 0 # New: for perplexity

        self.logger.info(f"Starting epoch {epoch+1} with {len(self.training_files)} batch files")

        # Progress bar
        pbar = tqdm(total=sum(len(self.cache.load_batches(f)) for f in self.training_files), desc=f"Epoch {epoch+1}")

        for file_idx, batch_file in enumerate(self.training_files):
            try:
                # Load batches from file
                batches = self.cache.load_batches(batch_file)

                for batch in batches:
                    # Move batch to device
                    input_ids = batch['input_ids'].to(self.device, non_blocking=True)
                    attention_mask = batch['attention_mask'].to(self.device, non_blocking=True)
                    labels = batch['labels'].to(self.device, non_blocking=True)

                    # Mixed precision forward pass
                    with autocast(enabled=self.train_config.get('use_amp', True)):
                        outputs = self.model(input_ids, attention_mask=attention_mask, labels=labels)
                        loss = outputs['loss']
                        logits = outputs['logits']

                    # Scale loss for gradient accumulation
                    loss = loss / self.grad_acc_steps
                    self.scaler.scale(loss).backward()

                    # Calculate accuracy and perplexity
                    accuracy = self.calculate_accuracy(logits, labels)
                    perplexity = torch.exp(outputs['loss']).item() if outputs['loss'] is not None else 0 # New: Perplexity

                    accumulated_loss += loss.item() * self.grad_acc_steps # Unscale loss for logging
                    accumulated_accuracy += accuracy
                    accumulated_perplexity += perplexity # New: Accumulate perplexity
                    micro_batch_count += 1

                    # Perform optimizer step only after accumulating gradients
                    if micro_batch_count % self.grad_acc_steps == 0:
                        # Gradient clipping
                        self.scaler.unscale_(self.optimizer)
                        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)

                        # Optimizer step
                        self.scaler.step(self.optimizer)
                        self.scaler.update()
                        self.optimizer.zero_grad()

                        # Scheduler step
                        if self.scheduler: # Only step if scheduler is initialized
                            self.scheduler.step()

                        self.global_step += 1 # Only increment global_step on actual optimization step

                        # Logging for optimization steps
                        if self.global_step % self.train_config.get('log_frequency', 100) == 0:
                            current_lr = self.optimizer.param_groups[0]['lr'] # Get current LR from scheduler
                            avg_accumulated_loss = accumulated_loss / self.grad_acc_steps
                            avg_accumulated_accuracy = accumulated_accuracy / self.grad_acc_steps
                            avg_accumulated_perplexity = accumulated_perplexity / self.grad_acc_steps # New

                            self.logger.info(
                                f"Step {self.global_step}: "
                                f"Loss: {avg_accumulated_loss:.4f}, "
                                f"Accuracy: {avg_accumulated_accuracy:.4f}, "
                                f"Perplexity: {avg_accumulated_perplexity:.2f}, " # New
                                f"LR: {current_lr:.2e}"
                            )

                            # TensorBoard logging
                            self.writer.add_scalar("train/loss", avg_accumulated_loss, self.global_step)
                            self.writer.add_scalar("train/accuracy", avg_accumulated_accuracy, self.global_step)
                            self.writer.add_scalar("train/perplexity", avg_accumulated_perplexity, self.global_step) # New
                            self.writer.add_scalar("train/learning_rate", current_lr, self.global_step)

                            # Memory usage
                            if torch.cuda.is_available():
                                memory_allocated = torch.cuda.memory_allocated() / 1024**3
                                self.writer.add_scalar("train/gpu_memory", memory_allocated, self.global_step)

                        # Reset accumulated metrics
                        total_loss += accumulated_loss
                        total_accuracy += accumulated_accuracy
                        total_perplexity += accumulated_perplexity # New
                        accumulated_loss = 0
                        accumulated_accuracy = 0
                        accumulated_perplexity = 0 # New

                    # Update progress bar every micro-batch
                    pbar.update(1)
                    pbar.set_postfix({
                        'micro_loss': f'{loss.item():.4f}', # Show micro-batch loss
                        'micro_acc': f'{accuracy:.4f}', # Show micro-batch accuracy
                        'micro_ppl': f'{perplexity:.2f}', # New: Show micro-batch perplexity
                        'opt_step': self.global_step # Show current optimization step
                    })

            except Exception as e:
                self.logger.error(f"Error processing batch file {batch_file}: {e}")
                # Ensure gradients are zeroed if an error occurs mid-accumulation cycle
                self.optimizer.zero_grad()
                accumulated_loss = 0
                accumulated_accuracy = 0
                accumulated_perplexity = 0 # New
                continue

        # Handle remaining accumulated gradients if not a perfect multiple
        if micro_batch_count % self.grad_acc_steps != 0:
            # Perform final optimizer step
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)
            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.optimizer.zero_grad()
            # Scheduler step
            if self.scheduler: # Only step if scheduler is initialized
                self.scheduler.step()
            self.global_step += 1

            total_loss += accumulated_loss
            total_accuracy += accumulated_accuracy
            total_perplexity += accumulated_perplexity # New

            current_lr = self.optimizer.param_groups[0]['lr'] # Get current LR from scheduler
            avg_accumulated_loss = accumulated_loss / (micro_batch_count % self.grad_acc_steps) # Average over remaining micro-batches
            avg_accumulated_accuracy = accumulated_accuracy / (micro_batch_count % self.grad_acc_steps)
            avg_accumulated_perplexity = accumulated_perplexity / (micro_batch_count % self.grad_acc_steps) # New

            self.logger.info(
                f"Final partial accumulation step {self.global_step}: "
                f"Loss: {avg_accumulated_loss:.4f}, "
                f"Accuracy: {avg_accumulated_accuracy:.4f}, "
                f"Perplexity: {avg_accumulated_perplexity:.2f}, " # New
                f"LR: {current_lr:.2e}"
            )

            self.writer.add_scalar("train/loss", avg_accumulated_loss, self.global_step)
            self.writer.add_scalar("train/accuracy", avg_accumulated_accuracy, self.global_step)
            self.writer.add_scalar("train/perplexity", avg_accumulated_perplexity, self.global_step) # New
            self.writer.add_scalar("train/learning_rate", current_lr, self.global_step)

            if torch.cuda.is_available():
                memory_allocated = torch.cuda.memory_allocated() / 1024**3
                self.writer.add_scalar("train/gpu_memory", memory_allocated, self.global_step)

        pbar.close()

        # Calculate epoch metrics based on optimization steps
        total_optimization_steps_in_epoch = self.global_step # Corrected to reflect actual optimization steps
        avg_loss = total_loss / (micro_batch_count / self.grad_acc_steps) if micro_batch_count > 0 else 0
        avg_accuracy = total_accuracy / (micro_batch_count / self.grad_acc_steps) if micro_batch_count > 0 else 0
        avg_perplexity = total_perplexity / (micro_batch_count / self.grad_acc_steps) if micro_batch_count > 0 else 0 # New

        self.logger.info(
            f"Epoch {epoch+1} completed: "
            f"Average Loss (per opt step): {avg_loss:.4f}, "
            f"Average Accuracy (per opt step): {avg_accuracy:.4f}, "
            f"Average Perplexity (per opt step): {avg_perplexity:.2f}, " # New
            f"Processed {micro_batch_count} micro-batches over {total_optimization_steps_in_epoch} optimization steps"
        )

        return avg_loss, avg_accuracy

    def validate(self, val_cache=None):
        """Run validation on validation cache if provided."""
        if val_cache is None:
            return None, None, None # New: Return None for perplexity too

        self.model.eval()
        total_loss = 0
        total_accuracy = 0
        total_perplexity = 0 # New
        batch_count = 0

        self.logger.info("Running validation...")

        with torch.no_grad():
            for batch in val_cache.iter_all_batches():
                try:
                    # Move batch to device
                    input_ids = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    labels = batch['labels'].to(self.device)

                    # Forward pass
                    with autocast(enabled=self.train_config.get('use_amp', True)):
                        outputs = self.model(input_ids, attention_mask=attention_mask, labels=labels)
                        loss = outputs['loss']
                        logits = outputs['logits']

                    # Calculate accuracy and perplexity
                    accuracy = self.calculate_accuracy(logits, labels)
                    perplexity = torch.exp(loss).item() if loss is not None else 0 # New

                    total_loss += loss.item()
                    total_accuracy += accuracy
                    total_perplexity += perplexity # New
                    batch_count += 1

                except Exception as e:
                    self.logger.error(f"Error in validation batch: {e}")
                    continue

        if batch_count == 0:
            return None, None, None # New: Return None for perplexity too

        avg_loss = total_loss / batch_count
        avg_accuracy = total_accuracy / batch_count
        avg_perplexity = total_perplexity / batch_count # New

        self.logger.info(
            f"Validation: Loss: {avg_loss:.4f}, Accuracy: {avg_accuracy:.4f}, "
            f"Perplexity: {avg_perplexity:.2f}, " # New
            f"Batches: {batch_count}"
        )

        # TensorBoard logging
        self.writer.add_scalar("val/loss", avg_loss, self.global_step)
        self.writer.add_scalar("val/accuracy", avg_accuracy, self.global_step)
        self.writer.add_scalar("val/perplexity", avg_perplexity, self.global_step) # New

        return avg_loss, avg_accuracy, avg_perplexity # Modified to return perplexity

    def train(self, val_cache=None):
        """Main training loop."""
        self.logger.info("Starting training...")
        self.logger.info(f"Model parameters: {sum(p.numel() for p in self.model.parameters()):,}")

        # Setup training components
        self.setup_training()

        # Load checkpoint if specified
        self.load_checkpoint()

        best_val_loss = float('inf')

        for epoch in range(self.epoch, self.train_config.get('num_epochs', 3)):
            self.epoch = epoch

            # Train for one epoch
            train_loss, train_accuracy = self.train_epoch(epoch)

            # Run validation
            val_loss, val_accuracy, val_perplexity = self.validate(val_cache) # Unpack perplexity

            # Save checkpoint
            checkpoint_file = save_checkpoint(
                self.train_config, epoch + 1, self.model, self.optimizer,
                self.scheduler, self.scaler, self.global_step, train_loss
            )
            self.logger.info(f"✅ Saved checkpoint: {checkpoint_file}")

            # Save best model
            if val_loss is not None and val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_file = get_weights_file_path(self.train_config, 'best')
                torch.save(self.model.state_dict(), best_model_file)
                self.logger.info(f"🏆 New best model saved: {best_model_file}")

        # Close TensorBoard writer
        self.writer.close()
        self.logger.info("Training completed!")

In [ ]:
cache_dir_path = '/content/drive/MyDrive/cache'
cache = SimpleBatchCache(cache_dir=cache_dir_path)
val_cache = None

## Trainng

In [ ]:

trainer = SimpleLLMTrainer(model, cache, train_config)
trainer.train(val_cache=val_cache)

/tmp/ipython-input-1313288217.py:77: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=self.train_config.get('use_amp', True))


Epoch 24:   0%|          | 0/5121 [00:00<?, ?it/s]

/tmp/ipython-input-1313288217.py:150: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.train_config.get('use_amp', True)):


# Validation

In [ ]:
checkpoint_path = "/content/drive/MyDrive/checkpoints/llm_epoch24.pt"

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
print("✅ Model loaded successfully from", checkpoint_path)

✅ Model loaded successfully from /content/drive/MyDrive/checkpoints/llm_epoch24.pt


In [ ]:
from transformers import RobertaTokenizer

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")


In [ ]:
query = "how many schedules in a indina constitution are there"
inputs = tokenizer.encode(query, return_tensors="pt")


In [ ]:
with torch.no_grad():
    inputs = inputs.to(device) # Move input tensor to the same device as the model
    outputs = model(inputs)

In [ ]:
outputs

{'logits': tensor([[[ -0.8286,   0.6031,   6.1772,  ...,  -7.4354, -11.4789,  -8.7477],
          [  0.0944,   0.6347,   3.4991,  ...,  -8.1862,  -8.6542,  -7.1043],
          [ -1.0965,   0.6635,   6.4872,  ...,  -6.0613,  -8.2729,  -9.5372],
          ...,
          [  1.5590,   0.6320,   5.3452,  ...,  -6.0020, -10.3186,  -8.5283],
          [ -2.6338,   0.6541,   3.6571,  ...,  -7.3681, -11.6342, -12.7763],
          [ -8.1203,   0.7055,  11.1841,  ...,  -9.1719,  -8.4277,  -7.9568]]],
        device='cuda:0'),
 'loss': None}

In [ ]:
predicted_token_ids = outputs['logits'].argmax(dim=-1)[0]
answer = tokenizer.decode(predicted_token_ids)
print("Model answer:", answer)

Model answer:   that.

 right on. to not is</s>


**Reasoning**:
I will create a new code cell and define the `generate_text` function as specified, implementing the autoregressive generation logic by tokenizing the prompt, iteratively calling the model, appending predicted tokens, and decoding the final sequence.



In [ ]:
def generate_text(model, tokenizer, prompt_text, max_new_tokens=200, device='cuda', eos_token_id=None, repetition_penalty=1.5):
    """
    Generates text autoregressively using the provided model and tokenizer.

    Args:
        model (nn.Module): The language model to use for generation.
        tokenizer: The tokenizer corresponding to the model.
        prompt_text (str): The initial text prompt to start generation from.
        max_new_tokens (int): The maximum number of new tokens to generate.
        device (str): The device to run the model on (e.g., 'cuda', 'cpu').
        eos_token_id (int, optional): The end-of-sequence token ID. Generation stops if this token is predicted.
        repetition_penalty (float): Penalty for repeating tokens. A value > 1.0 increases the penalty for repeated tokens.

    Returns:
        str: The generated text.
    """
    model.eval() # Set the model to evaluation mode

    # 3. Tokenize the prompt_text to get initial input_ids.
    input_ids = tokenizer.encode(prompt_text, return_tensors="pt")

    # Important: If the tokenizer automatically adds EOS, remove it for generation
    # so the model can generate it naturally. This handles cases like RoBERTa.
    if input_ids[0, -1] == tokenizer.eos_token_id:
        input_ids = input_ids[:, :-1]

    # 4. Move the input_ids tensor to the specified device.
    input_ids = input_ids.to(device)

    generated_ids = input_ids

    # Define special tokens to exclude from repetition penalty
    special_tokens_ids = set(tokenizer.all_special_ids) if hasattr(tokenizer, 'all_special_ids') else set()
    # Add BOS token explicitly
    if tokenizer.bos_token_id is not None:
        special_tokens_ids.add(tokenizer.bos_token_id)
    # Add PAD token explicitly
    if tokenizer.pad_token_id is not None:
        special_tokens_ids.add(tokenizer.pad_token_id)

    # 5. Start a loop that runs for max_new_tokens iterations.
    for _ in range(max_new_tokens):
        # 6. Within the loop, perform a forward pass through the model with the current input_ids to get logits.
        with torch.no_grad():
            outputs = model(generated_ids)
            logits = outputs['logits']

        # Get logits for the last token in the sequence
        next_token_logits = logits[:, -1, :]

        # Apply repetition penalty, but skip special tokens
        if repetition_penalty != 1.0:
            for token_id_in_seq in generated_ids[0]:
                token_id_val = token_id_in_seq.item()
                if token_id_val in special_tokens_ids: # Skip special tokens from penalty
                    continue
                # Apply penalty: if logit is positive, divide; if negative, multiply
                if next_token_logits[0, token_id_val] < 0:
                    next_token_logits[0, token_id_val] *= repetition_penalty
                else:
                    next_token_logits[0, token_id_val] /= repetition_penalty

        # 7. Identify the next_token_id by finding the token with the highest logit.
        next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)

        # 8. Check if the next_token_id is the eos_token_id or pad_token_id. If it is, break the loop.
        if (eos_token_id is not None and next_token_id.item() == eos_token_id) or \
           (tokenizer.pad_token_id is not None and next_token_id.item() == tokenizer.pad_token_id):
            break

        # 9. Append the next_token_id to the input_ids tensor.
        generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)

    # 10. After the loop, decode the final input_ids tensor back into text using the tokenizer.
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    # 11. Return the generated text.
    return generated_text

In [ ]:
query = " tell me about supreme court of india"
generated_answer = generate_text(model, tokenizer, query, device=device, repetition_penalty=1.2, eos_token_id=tokenizer.eos_token_id)
print("Generated answer stored.")

Generated answer stored.


In [ ]:
generated_answer

' tell me about supreme court of india, and the National Assembly of India. The first time was a result in the United States, which was not to be a major role in the country. In addition to the Indian state government\'s first time, he had been used for a new government. The term \\"the most important role\\" was not considered an independent of the United Kingdom and was \\"a-speaking-day-largest city\\". On October 13, 2024, it would have been made by a new government\'s decision on November 4.5 million people were also known as well as \\"The first day\\" (or \\"unachal) and \\"Hindu\\", which had been used for its own political parties that he would be used to be held at least one of his death toll with an independent number of their own government. This is also called for this period that they are not only one or more than two years before it is not only in order to have been described as part in their own case against them from other'

In [ ]:
generated_answer

' tell me about supreme court of india, and the National Assembly of India. The first time was a result in the United States, which was not to be a major role in the country. In addition to the Indian state government\'s first time, he had been used for a new government. The term \\"the most important role\\" was not considered an independent of the United Kingdom and was \\"a-speaking-day-largest city\\". On October 13, 2024, it would have been made by a new government\'s decision on November 4.5 million people were also known as well as \\"The first day\\" (or \\"unachal) and \\"Hindu\\", which had been used for its own political parties that he would be used to be held at least one of his death toll with an independent number of their own government. This is also called for this period that they are not only one or more than two years before it is not only in order to have been described as part in their own case against them from other'

In [ ]:
generated_answer

'which is higher than the supreme court of india, the first time. The first time in the first time to the first time to a \\"the \\"The first time\\" in the first time. The first time, a \\"the first time to be used by a \\"The first time\\" and the first two-year-in-in-year-old, which is not only one of the first time. In addition to be used as a new government of the first time.\\n\\ECTION: The first time, which was an important of the first two years after his own, but it is an \\"to-day-based\\" or a \\"I\\", and the first time in a new government. In this period, he was not have been used for their own. This is also known as well as well as an \\"the \\"a\\" or \\"The first time\\" (1851) and\\" (1956), which are not only one of its own own \\"A\\" (1823), with a new government'

## info


In [ ]:
! pip install torchinfo

In [ ]:
from torchinfo import summary

In [ ]:
summary(model, input_size=(1, 128), dtypes=[torch.long])

Layer (type:depth-idx)                                  Output Shape              Param #
SimpleLLM                                               --                        --
├─EmbeddingLayer: 1-1                                   [1, 128, 512]             --
│    └─Embedding: 2-1                                   [1, 128, 512]             25,735,680
│    └─Dropout: 2-2                                     [1, 128, 512]             --
├─MultiTransformerBlock: 1-2                            [1, 128, 512]             --
│    └─ModuleList: 2-3                                  --                        --
│    │    └─TransformerBlock: 3-1                       [1, 128, 512]             3,933,312
│    │    └─TransformerBlock: 3-2                       [1, 128, 512]             3,933,312
│    │    └─TransformerBlock: 3-3                       [1, 128, 512]             3,933,312
│    │    └─TransformerBlock: 3-4                       [1, 128, 512]             3,933,312
│    │    └─TransformerB

In [ ]:
print("\n--- Model Parameter Shapes ---")
for name, param in model.named_parameters():
    print(f"Parameter: {name}, Shape: {param.shape}")


--- Model Parameter Shapes ---
Parameter: embedding.token_embedding.weight, Shape: torch.Size([50265, 512])
Parameter: transformer.layers.0.attn_norm.gamma, Shape: torch.Size([512])
Parameter: transformer.layers.0.attention.q_proj.weight, Shape: torch.Size([512, 512])
Parameter: transformer.layers.0.attention.k_proj.weight, Shape: torch.Size([256, 512])
Parameter: transformer.layers.0.attention.v_proj.weight, Shape: torch.Size([256, 512])
Parameter: transformer.layers.0.attention.o_proj.weight, Shape: torch.Size([512, 512])
Parameter: transformer.layers.0.attention.q_norm.gamma, Shape: torch.Size([64])
Parameter: transformer.layers.0.attention.k_norm.gamma, Shape: torch.Size([64])
Parameter: transformer.layers.0.ffn_norm.gamma, Shape: torch.Size([512])
Parameter: transformer.layers.0.ffn.gate_proj.weight, Shape: torch.Size([2048, 512])
Parameter: transformer.layers.0.ffn.up_proj.weight, Shape: torch.Size([2048, 512])
Parameter: transformer.layers.0.ffn.down_proj.weight, Shape: torch.S